In [2]:
# =============================================================================
# Zelle 01 – Setup & Daten laden (Preprocessing Modell B)
# =============================================================================
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from viz_config_v2 import apply_store44_style, save_figure, COLOR_GOLD, COLOR_BLUE, COLOR_GREEN, COLOR_TEXT_MUTED, BOXPLOT_STYLE

SEED = 42
apply_store44_style()

df_b = pd.read_csv("../data/raw/model_b_raw.csv")

X_B_MERKMALE = ["material_mfr", "dn_ziel", "wandstaerke_soll", "dickentoleranz",
                 "wandtyp", "produktionsgeschwindigkeit_soll", "ovalitaet_anforderung"]
Y_B_MERKMALE = ["schneckendrehzahl", "massetemperatur", "duesenspalt",
                 "vakuumniveau", "innenluftdruck", "kuehlwassertemperatur"]

# --- Preprocessing-Empfehlungen aus Notebook 09 laden (Single Source of Truth) ---
preprocessing_empfehlung = pd.read_csv("../reports/tables/09_finale_preprocessing_empfehlung_model_b.csv")

print(f"Datensatz: {df_b.shape}")
print(f"X_B ({len(X_B_MERKMALE)}): {X_B_MERKMALE}")
print(f"Y_B ({len(Y_B_MERKMALE)}): {Y_B_MERKMALE}")
print(f"\nPreprocessing-Empfehlungen aus Notebook 09 geladen:")
print(preprocessing_empfehlung[["merkmal", "yeo_johnson_empfohlen", "skalierung"]].to_string(index=False))

Datensatz: (2000, 14)
X_B (7): ['material_mfr', 'dn_ziel', 'wandstaerke_soll', 'dickentoleranz', 'wandtyp', 'produktionsgeschwindigkeit_soll', 'ovalitaet_anforderung']
Y_B (6): ['schneckendrehzahl', 'massetemperatur', 'duesenspalt', 'vakuumniveau', 'innenluftdruck', 'kuehlwassertemperatur']

Preprocessing-Empfehlungen aus Notebook 09 geladen:
                        merkmal  yeo_johnson_empfohlen     skalierung
                   material_mfr                  False StandardScaler
                        dn_ziel                   True StandardScaler
               wandstaerke_soll                   True StandardScaler
                 dickentoleranz                  False StandardScaler
produktionsgeschwindigkeit_soll                  False StandardScaler
          ovalitaet_anforderung                  False StandardScaler
              schneckendrehzahl                   True StandardScaler
                massetemperatur                  False StandardScaler
                    duese

In [3]:
# =============================================================================
# Zelle 02 – Encoding: wandtyp (einzige kategoriale X_B-Variable)
# =============================================================================
# Analog Modell A: One-Hot-Encoding mit drop_first=True (Dummy-Variable-
# Trap vermeiden, siehe Notebook 05 Lehre). Nur 1 kategoriale Variable
# bei Modell B (kein kalibriermechanismus mehr, siehe Root-Cause-Fix).
# =============================================================================
df_b_encoded = pd.get_dummies(df_b, columns=["wandtyp"], prefix="wandtyp", drop_first=True)

neue_spalte = [c for c in df_b_encoded.columns if c.startswith("wandtyp_")]
print(f"Neue Encoding-Spalte: {neue_spalte}")
print(df_b_encoded[neue_spalte].sum())

df_b = df_b_encoded
X_B_MERKMALE_ENCODED = [c for c in X_B_MERKMALE if c != "wandtyp"] + neue_spalte
print(f"\nX_B_MERKMALE (encoded): {X_B_MERKMALE_ENCODED}")

Neue Encoding-Spalte: ['wandtyp_einwandig']
wandtyp_einwandig    1475
dtype: int64

X_B_MERKMALE (encoded): ['material_mfr', 'dn_ziel', 'wandstaerke_soll', 'dickentoleranz', 'produktionsgeschwindigkeit_soll', 'ovalitaet_anforderung', 'wandtyp_einwandig']


In [4]:
# =============================================================================
# Zelle 03 – DN-Wandstaerke-Residualizer (leakage-sicherer Custom-Transformer)
# =============================================================================
# Analog zu Modell A's DNResidualizer (src/preprocessing.py): berechnet
# den Anteil von wandstaerke_soll, der NICHT durch dn_ziel erklaert wird.
# fit() lernt nur aus Trainingsdaten (bei spaeterer Pipeline-Nutzung),
# strukturell leakage-sicher.
# =============================================================================
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LinearRegression

class WandstaerkeDNResidualizer(BaseEstimator, TransformerMixin):
    """Berechnet DN-bereinigte Residuen fuer wandstaerke_soll."""

    def fit(self, X, y=None):
        X = pd.DataFrame(X, columns=["dn_ziel", "wandstaerke_soll"]) if not isinstance(X, pd.DataFrame) else X
        self.regression_ = LinearRegression().fit(X[["dn_ziel"]], X["wandstaerke_soll"])
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=["dn_ziel", "wandstaerke_soll"]) if not isinstance(X, pd.DataFrame) else X
        vorhersage = self.regression_.predict(X[["dn_ziel"]])
        residuum = pd.DataFrame(index=X.index)
        residuum["wandstaerke_soll_dn_bereinigt"] = X["wandstaerke_soll"].values - vorhersage
        return residuum

# --- Kurzer Test ---
test_residualizer = WandstaerkeDNResidualizer()
test_residualizer.fit(df_b[["dn_ziel", "wandstaerke_soll"]])
test_residuen = test_residualizer.transform(df_b[["dn_ziel", "wandstaerke_soll"]])
print(f"Residuen berechnet: Mean={test_residuen['wandstaerke_soll_dn_bereinigt'].mean():.4f} (sollte ~0 sein), Std={test_residuen['wandstaerke_soll_dn_bereinigt'].std():.4f}")
print(f"\nKorrelation Residuum zu DN (sollte ~0 sein): {test_residuen['wandstaerke_soll_dn_bereinigt'].corr(df_b['dn_ziel']):.4f}")

Residuen berechnet: Mean=-0.0000 (sollte ~0 sein), Std=0.1501

Korrelation Residuum zu DN (sollte ~0 sein): 0.0000
